In [1]:
import os
import pandas as pd
import geopandas as gpd
import geobr

# Importar as funções desenvolvidas nos módulos da pasta src/soja_clima/
from src.soja_clima.qualidade import filtrar_estacoes_fora_do_brasil
from src.soja_clima.sidra import ler_sidra_soja, agregar_soja_para_microrregiao
from src.soja_clima.geo import construir_mapa_estacao_microrregiao
from src.soja_clima.agregacao import (
    agregar_diario_para_estacao_safra,
    agregar_espacial_para_microrregiao,
)
from src.soja_clima.merge import montar_base_final

In [2]:
# ==============================================================================
# 1. CONFIGURAÇÃO DOS CAMINHOS
# ==============================================================================
BASE_DIR = "/Users/lauranannetti/Documents/PCD2/projeto-pcd2"

CAMINHO_CLIMA = os.path.join(BASE_DIR, "data/processed/clima_diario.parquet")
CAMINHO_SIDRA = os.path.join(BASE_DIR, "data/raw/tabela1612.csv")
CAMINHO_SAIDA = os.path.join(BASE_DIR, "data/processed/base_final_modelagem.csv")

print("--- A iniciar o Pipeline de Integração ---")


--- A iniciar o Pipeline de Integração ---


In [3]:
# ==============================================================================
# 2. CARREGAMENTO E QUALIDADE DOS DADOS CLIMÁTICOS
# ==============================================================================
print("\n[1/6] A carregar e a filtrar dados climáticos...")
diario = pd.read_parquet(CAMINHO_CLIMA)
diario["data"] = pd.to_datetime(diario["data"])

# Filtrar estações localizadas fora da caixa geográfica do Brasil
diario, estacoes_removidas = filtrar_estacoes_fora_do_brasil(diario)
print(f"   ✓ Dados climáticos prontos. Registos válidos: {len(diario):,}")



[1/6] A carregar e a filtrar dados climáticos...
   ✓ Dados climáticos prontos. Registos válidos: 2,764,708


In [5]:
# ==============================================================================
# 3. LEITURA E PREPARAÇÃO DOS DADOS DE PRODUÇÃO DE SOJA (SIDRA)
# ==============================================================================
print("\n[2/6] A ler e a processar dados da Tabela 1612 do SIDRA...")
df_soja_mun = ler_sidra_soja(CAMINHO_SIDRA)

print("   - Descarregando malhas do geobr e associando municípios às microrregiões...")
mun_br = geobr.read_municipality(year=2020)
microrregioes = geobr.read_micro_region(year=2020)

# Join espacial: usa o centroide de cada município para identificar sua microrregião
mun_br_centroids = mun_br.copy()
mun_br_centroids["geometry"] = mun_br_centroids.geometry.centroid

mun_com_micro = gpd.sjoin(
    mun_br_centroids,
    microrregioes[["code_micro", "name_micro", "geometry"]],
    how="left",
    predicate="within"
)

# Normalizar texto dos municípios para garantir o correspondência exata no merge
df_soja_mun["mun_join"] = (
    df_soja_mun["municipio"]
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)
mun_com_micro["mun_join"] = (
    mun_com_micro["name_muni"]
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

# Associar o código da microrregião (code_micro) aos dados municipais do SIDRA
df_soja_mun_com_micro = df_soja_mun.merge(
    mun_com_micro[["mun_join", "abbrev_state", "code_micro", "name_micro"]],
    left_on=["mun_join", "uf"],
    right_on=["mun_join", "abbrev_state"],
    how="inner",
).drop(columns=["mun_join", "abbrev_state"])

# Agregar a produção agrícola ao nível de microrregião e recalcular o rendimento (kg/ha)
df_soja_micro = agregar_soja_para_microrregiao(df_soja_mun_com_micro)
print(f"   ✓ Produção agregada por microrregião. Linhas: {len(df_soja_micro):,}")


[2/6] A ler e a processar dados da Tabela 1612 do SIDRA...
   - Descarregando malhas do geobr e associando municípios às microrregiões...


microregions_2020_simplified.parquet: 100%|██████████| 7.91M/7.91M [00:00<00:00, 14.4MB/s]
/var/folders/j3/kppygfjs7xj9nbzbwjj9cjc80000gn/T/ipykernel_29805/1346740053.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  mun_br_centroids["geometry"] = mun_br_centroids.geometry.centroid


   ✓ Produção agregada por microrregião. Linhas: 14,508


In [7]:
# ==============================================================================
# 4. MAPEAMENTO ESPACIAL (ESTAÇÕES METEOROLÓGICAS -> MICRORREGIÕES)
# ==============================================================================
print("\n[3/6] A descarregar malha de microrregiões e a efetuar o join espacial...")
microrregioes = geobr.read_micro_region(year=2020)

# Mapear cada estação à respetiva microrregião (com tratamento para pontos fora de polígonos)
mapa_estacoes = construir_mapa_estacao_microrregiao(diario, microrregioes)

# Incorporar o 'code_micro' ao dataframe diário do clima
diario_com_micro = diario.merge(
    mapa_estacoes[["codigo_estacao", "code_micro"]],
    on="codigo_estacao",
    how="inner",
)
print("   ✓ Join espacial das estações concluído.")


[3/6] A descarregar malha de microrregiões e a efetuar o join espacial...
547 estação(ões) com coordenadas divergentes entre arquivos -- usando a mediana das coordenadas.
162 de 558 microrregiões não têm nenhuma estação dentro delas.
5 microrregião(ões) sem substituta dentro de 150km -- ficarão sem clima.
   ✓ Join espacial das estações concluído.


In [8]:
# ==============================================================================
# 5. AGREGAÇÃO TEMPORAL E ESPACIAL DO CLIMA
# ==============================================================================
print("\n[4/6] A agregar clima no tempo (Diário -> Estação/Safra por fases fenológicas)...")
estacao_safra = agregar_diario_para_estacao_safra(diario_com_micro, usar_fases=True)

print("\n[5/6] A agregar clima no espaço (Estação/Safra -> Microrregião/Safra)...")
clima_micro_safra = agregar_espacial_para_microrregiao(estacao_safra)
print(f"   ✓ Clima agregador por microrregião e safra. Linhas: {len(clima_micro_safra):,}")


[4/6] A agregar clima no tempo (Diário -> Estação/Safra por fases fenológicas)...

[5/6] A agregar clima no espaço (Estação/Safra -> Microrregião/Safra)...
   ✓ Clima agregador por microrregião e safra. Linhas: 8,933


In [9]:
# ==============================================================================
# 6. MERGE FINAL E EXPORTAÇÃO
# ==============================================================================
print("\n[6/6] A realizar o merge final entre Clima e Produção de Soja...")
base_final = montar_base_final(clima_micro_safra, df_soja_micro)

# Assegurar que a pasta de destino existe e guardar o ficheiro CSV
os.makedirs(os.path.dirname(CAMINHO_SAIDA), exist_ok=True)
base_final.to_csv(CAMINHO_SAIDA, index=False)

print("\n==================================================================")
print("--- Pipeline concluído com sucesso! ---")
print(f"Ficheiro guardado em: {CAMINHO_SAIDA}")
print(f"Dimensão da base final pronta para modelagem: {base_final.shape}")
print("==================================================================")


[6/6] A realizar o merge final entre Clima e Produção de Soja...

--- Pipeline concluído com sucesso! ---
Ficheiro guardado em: /Users/lauranannetti/Documents/PCD2/projeto-pcd2/data/processed/base_final_modelagem.csv
Dimensão da base final pronta para modelagem: (8933, 45)
